In [ ]:
!pip install vllm

In [ ]:
!huggingface-cli login

In [ ]:
import vllm

In [ ]:
print(vllm.__version__)

In [ ]:
!nvidia-smi

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU. Go to Runtime -> Change runtime type -> T4 GPU')
gpu_info = r.stdout.strip()
print('GPU:', gpu_info)
free_mb = int(gpu_info.split(',')[2].strip().replace(' MiB',''))
print('VRAM free:', free_mb, 'MB', '-- OK' if free_mb > 10000 else '-- LOW, restart runtime')


In [ ]:
import subprocess
def run(cmd, desc):
    print(f'Installing {desc}...', end=' ', flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('FAILED'); print(r.stderr[-2000:]); raise RuntimeError(desc)
    print('done')
run('pip install -q peft==0.10.0 accelerate', 'PEFT + Accelerate')
run('pip install -q openai httpx rich', 'OpenAI client + rich')
print('All done.')

In [ ]:
!pip install transformers==4.40.2

# Create LoRA Adpters

In [ ]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_CODE_PATH = '/content/adapters/code-assistant'
ADAPTER_SUMM_PATH = '/content/adapters/summarizer'
os.makedirs(ADAPTER_CODE_PATH, exist_ok=True)
os.makedirs(ADAPTER_SUMM_PATH, exist_ok=True)

print('Loading base model')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='cpu')
print(f'Loaded: {sum(p.numel() for p in base_model.parameters())/1e9:.2f}B params')

# Adapter 1: Code Assistant (rank=8)
print('Creating code-assistant adapter (rank=8)')
m1 = get_peft_model(base_model, LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, target_modules=['q_proj','v_proj'], lora_dropout=0.05, bias='none'))
tr, tot = m1.get_nb_trainable_parameters()
print(f'  Trainable: {tr/1e6:.2f}M / {tot/1e9:.2f}B ({100*tr/tot:.2f}%)')
m1.save_pretrained(ADAPTER_CODE_PATH); tokenizer.save_pretrained(ADAPTER_CODE_PATH)
del m1

# Adapter 2: Summarizer (rank=16, also adapts k_proj)
print('Creating summarizer adapter (rank=16)')
m2 = get_peft_model(base_model, LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, target_modules=['q_proj','v_proj','k_proj'], lora_dropout=0.05, bias='none'))
tr, tot = m2.get_nb_trainable_parameters()
print(f'  Trainable: {tr/1e6:.2f}M / {tot/1e9:.2f}B ({100*tr/tot:.2f}%)')
m2.save_pretrained(ADAPTER_SUMM_PATH); tokenizer.save_pretrained(ADAPTER_SUMM_PATH)
del m2, base_model

import gc; gc.collect(); torch.cuda.empty_cache()

sz = lambda p: sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(p) for f in fs)/1e6
print(f'Adapter sizes: code={sz(ADAPTER_CODE_PATH):.1f}MB  summarizer={sz(ADAPTER_SUMM_PATH):.1f}MB')
print('Both adapters share the same base model in GPU memory at serve time.')

In [ ]:
!pip install -q --upgrade transformers vllm

In [ ]:
import transformers, vllm

print("Transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

### **Launch vLLM**

In [ ]:
import subprocess, time, httpx

BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_CODE = '/content/adapters/code-assistant'
ADAPTER_SUMM = '/content/adapters/summarizer'
LOG_FILE     = '/content/vllm_server.log'

cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', BASE_MODEL,
    '--port', '8000',
    '--dtype', 'float16',
    '--max-model-len', '1024',
    '--gpu-memory-utilization', '0.65',
    '--enforce-eager',
    '--enable-lora',
    '--max-lora-rank', '16',
    '--lora-modules',
        f'code-assistant={ADAPTER_CODE}',
        f'summarizer={ADAPTER_SUMM}',
]

print('Starting vLLM...')
log_f = open(LOG_FILE, 'w')
server_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)

for attempt in range(60):
    time.sleep(2)
    try:
        if httpx.get('http://localhost:8000/health', timeout=3).status_code == 200:
            print(f'Server ready after {(attempt+1)*2}s')
            break
    except Exception:
        pass
    if attempt % 5 == 4:
        print(f'  Still loading... ({(attempt+1)*2}s)')
else:
    with open(LOG_FILE) as f:
        print(''.join(f.readlines()[-30:]))
    raise RuntimeError('vLLM failed to start')

models = httpx.get('http://localhost:8000/v1/models').json()
print('Registered adapters:')
for m in models['data']:
    print(f'  {m["id"]}')


Router

In [ ]:
from openai import OpenAI
import re, time

client = OpenAI(base_url='http://localhost:8000/v1', api_key='not-needed')

CONFIGS = {
    'code-assistant': {
        'system': 'You are an expert programming assistant. Always respond with clean, well-commented code. Briefly explain what it does first.',
        'temperature': 0.2, 'max_tokens': 400,
    },
    'summarizer': {
        'system': 'You are a concise summarization assistant. Extract key points as a tight structured summary with bullet points.',
        'temperature': 0.4, 'max_tokens': 300,
    },
}

class LoRARouter:
    CODE_RE = re.compile(r'\b(write|code|implement|function|class|script|python|javascript|algorithm|program|debug|fix|refactor|sql|regex)\b', re.I)

    def route(self, msg):
        return 'code-assistant' if self.CODE_RE.search(msg) else 'summarizer'

    def complete(self, msg, adapter=None):
        adapter = adapter or self.route(msg)
        cfg = CONFIGS[adapter]
        t0 = time.perf_counter()
        resp = client.chat.completions.create(
            model=adapter,
            messages=[{'role':'system','content':cfg['system']},{'role':'user','content':msg}],
            temperature=cfg['temperature'], max_tokens=cfg['max_tokens'],
        )
        lat = time.perf_counter() - t0
        toks = resp.usage.completion_tokens
        return {'adapter': adapter, 'content': resp.choices[0].message.content,
                'tokens': toks, 'latency_s': round(lat,2), 'tok_per_s': round(toks/lat,1)}

router = LoRARouter()
print('Router ready. Tests:')
for msg in ['Write a Python function','Summarize this article','implement binary search']:
    print(f'  "{msg}" -> {router.route(msg)}')

**Comparison of both Adapters**

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.columns import Columns
console = Console()

PROMPTS = [
    'Explain how a hash table works.',
    'What is gradient descent?',
    'Describe the difference between TCP and UDP.',
]

console.print('[bold cyan]Multi-LoRA Adapter Comparison[/bold cyan]\n')
for i, prompt in enumerate(PROMPTS, 1):
    console.print(f'[bold yellow]Prompt {i}:[/bold yellow] {prompt}')
    cr = router.complete(prompt, adapter='code-assistant')
    sr = router.complete(prompt, adapter='summarizer')
    console.print(Columns([
        Panel(cr['content'], title=f'[green]code-assistant[/green] {cr["latency_s"]}s', border_style='green', padding=(1,2)),
        Panel(sr['content'], title=f'[blue]summarizer[/blue] {sr["latency_s"]}s', border_style='blue', padding=(1,2)),
    ], equal=True))
    console.print()
console.print('[bold green]Both adapters served from one TinyLlama-1.1B instance.[/bold green]')


Auto-routing

In [ ]:
from rich.console import Console
from rich.panel import Panel
console = Console()

MIXED = [
    'Write a Python function for longest common subsequence.',
    'Summarize the key ideas behind transformer architecture.',
    'Implement a rate limiter using a sliding window.',
    'What are the main takeaways from Attention Is All You Need?',
]

console.print('[bold cyan]Auto-Router Demo[/bold cyan]\n')
for prompt in MIXED:
    r = router.complete(prompt)
    color = 'green' if r['adapter'] == 'code-assistant' else 'blue'
    console.print(Panel(f'[dim]{prompt}[/dim]\n\n{r["content"]}',
        title=f'[{color}]-> {r["adapter"]}[/{color}] {r["latency_s"]}s', border_style=color))
    console.print()

Benchmark

In [ ]:
import time
from rich.console import Console
from rich.table import Table
from rich import box
console = Console()

BENCH = [
    ('Write a Python hello world.', 'code-assistant'),
    ('Summarize: AI is transforming software development.', 'summarizer'),
    ('Implement bubble sort in Python.', 'code-assistant'),
    ('Summarize the concept of neural networks.', 'summarizer'),
    ('Write a string reverse function.', 'code-assistant'),
    ('Summarize: LoRA reduces fine-tuning cost.', 'summarizer'),
]

console.print(f'[bold cyan]Benchmark: {len(BENCH)} requests, alternating adapters[/bold cyan]\n')
results = []
t0 = time.perf_counter()
for prompt, adapter in BENCH:
    r = router.complete(prompt, adapter=adapter)
    results.append(r)
    console.print(f'  [{adapter}] {r["latency_s"]}s / {r["tok_per_s"]} tok/s')

total_time = time.perf_counter() - t0
code_rs = [r for r in results if r['adapter']=='code-assistant']
summ_rs = [r for r in results if r['adapter']=='summarizer']

t = Table(box=box.ROUNDED, title='Benchmark Summary')
t.add_column('Metric', style='cyan'); t.add_column('Value')
t.add_row('Total requests', str(len(results)))
t.add_row('Wall time', f'{total_time:.1f}s')
t.add_row('Total tokens', str(sum(r['tokens'] for r in results)))
t.add_row('Avg latency', f'{sum(r["latency_s"] for r in results)/len(results):.2f}s')
t.add_row('code-assistant avg', f'{sum(r["latency_s"] for r in code_rs)/len(code_rs):.2f}s')
t.add_row('summarizer avg', f'{sum(r["latency_s"] for r in summ_rs)/len(summ_rs):.2f}s')
t.add_row('Adapter switches', str(len(results)-1))
console.print(t)

In [ ]:
import signal
server_proc.send_signal(signal.SIGTERM)
server_proc.wait(timeout=10)
import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('Server stopped. GPU memory freed.')